In [1]:
import numpy as np
import torch
import os
import logging
import yaml
import pandas as pd
import sys
import matplotlib.pyplot as plt
import glob
import awkward as ak
import pandas as pd
import joblib  # For saving the scaler
from sklearn.model_selection import train_test_split

/home/aegis/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [ ]:
helpers_path = os.path.join('/home/aegis/Titan1/NRAD/data/model_scripts')
sys.path.insert(0, os.path.abspath(helpers_path))
from Classifier import Classifier
from SimpleMAF import SimpleMAF

mc_path = "/home/aegis/Titan1/NRAD/data/Regions"
data_path = "/home/aegis/Titan1/NRAD/data/Regions_data"

# MC Processes
mc_list = ["Diboson", "Multijet", "Single_top", "ttbar", "Wjets", "Zjets"]

In [23]:
MC_CR_emu_path = glob.glob(os.path.join(mc_path, f"CR_emu", "*.parquet"))
MC_CR0L_path = glob.glob(os.path.join(mc_path, "CR0L", "*.parquet"))
MC_CR1ele_path = glob.glob(os.path.join(mc_path, "CR1ele", "*.parquet"))
MC_CR1eleb_path = glob.glob(os.path.join(mc_path, "CR1eleb", "*.parquet"))
MC_CR1mu_path = glob.glob(os.path.join(mc_path, "CR1mu", "*.parquet"))
MC_CR1mub_path = glob.glob(os.path.join(mc_path, "CR1mub", "*.parquet"))
MC_CR2ele_path = glob.glob(os.path.join(mc_path, "CR2ele", "*.parquet"))
MC_CR2mu_path = glob.glob(os.path.join(mc_path, "CR2mu", "*.parquet"))

print("CR_emu files:", len(MC_CR_emu_path))
print("CR0L files:", len(MC_CR0L_path))
print("CR1ele files:", len(MC_CR1ele_path))
print("CR1eleb files:", len(MC_CR1eleb_path))
print("CR1mu files:", len(MC_CR1mu_path))
print("CR1mub files:", len(MC_CR1mub_path))
print("CR2ele files:", len(MC_CR2ele_path))
print("CR2mu files:", len(MC_CR2mu_path))

CR_emu files: 4392
CR0L files: 5353
CR1ele files: 3799
CR1eleb files: 3065
CR1mu files: 5217
CR1mub files: 4697
CR2ele files: 1334
CR2mu files: 3895


### Process CR_emu

In [22]:
print("CR_emu files:", len(MC_CR_emu_path))

Diboson_CR_emu_path = glob.glob(os.path.join(mc_path, f"CR_emu", "Diboson*.parquet"))
Multijet_CR_emu_path = glob.glob(os.path.join(mc_path, f"CR_emu", "Multijet*.parquet"))
Single_top_CR_emu_path = glob.glob(os.path.join(mc_path, f"CR_emu", "Single_top*.parquet"))
ttbar_CR_emu_path = glob.glob(os.path.join(mc_path, f"CR_emu", "ttbar*.parquet"))
Wjets_CR_emu_path = glob.glob(os.path.join(mc_path, f"CR_emu", "Wjets*.parquet"))
Zjets_CR_emu_path = glob.glob(os.path.join(mc_path, f"CR_emu", "Zjets*.parquet"))

print("Diboson CR_emu files:", len(Diboson_CR_emu_path))
print("Multijet CR_emu files:", len(Multijet_CR_emu_path))
print("Single_top CR_emu files:", len(Single_top_CR_emu_path))
print("ttbar CR_emu files:", len(ttbar_CR_emu_path))
print("Wjets CR_emu files:", len(Wjets_CR_emu_path))
print("Zjets CR_emu files:", len(Zjets_CR_emu_path))   

print()
print("="*10,"CR_emu files check:", "="*10)
if len(MC_CR_emu_path) - len(Diboson_CR_emu_path) - len(Multijet_CR_emu_path) - len(Single_top_CR_emu_path) - len(ttbar_CR_emu_path) - len(Wjets_CR_emu_path) - len(Zjets_CR_emu_path) == 0:
    print("All files are accounted for in CR_emu.")
else:
    print("Error: Some files in CR_emu are not accounted for by the individual processes.")

CR_emu files: 4392
Diboson CR_emu files: 20
Multijet CR_emu files: 228
Single_top CR_emu files: 16
ttbar CR_emu files: 124
Wjets CR_emu files: 3633
Zjets CR_emu files: 371

========== CR_emu files check: ==========
All files are accounted for in CR_emu.


In [24]:
# process diboson files
diboson_dfs = []
for file in Diboson_CR_emu_path:
    df = pd.read_parquet(file)
    df['process'] = 'Diboson'
    diboson_dfs.append(df)
diboson_df = pd.concat(diboson_dfs, ignore_index=True)

In [26]:
diboson_df.columns

Index(['MET_Core_AnalysisMETAuxDyn_mpx', 'MET_Core_AnalysisMETAuxDyn_mpy',
       'MET_Core_AnalysisMETAuxDyn_sumet', 'EventInfoAuxDyn_mcEventWeights',
       'AnalysisJetsAuxDyn_pt', 'AnalysisJetsAuxDyn_eta',
       'AnalysisJetsAuxDyn_phi', 'AnalysisJetsAuxDyn_NNJvtPass',
       'AnalysisLargeRJetsAuxDyn_pt', 'AnalysisLargeRJetsAuxDyn_eta',
       'AnalysisLargeRJetsAuxDyn_phi', 'AnalysisLargeRJetsAuxDyn_m',
       'AnalysisLargeRJetsAuxDyn_Tau1_wta',
       'AnalysisLargeRJetsAuxDyn_Tau2_wta',
       'AnalysisLargeRJetsAuxDyn_Tau3_wta',
       'AnalysisElectronsAuxDyn_DFCommonElectronsLHTight',
       'AnalysisMuonsAuxDyn_muonType', 'AnalysisMuonsAuxDyn_quality',
       'AnalysisTauJetsAuxDyn_JetDeepSetTight',
       'BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu',
       'BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc',
       'BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb', 'weight_phys',
       'met_recalc_pt', 'met_recalc_phi', 'is_bjet', 'n_bjets', 'n_ele',
       'n_mu', 'process'],
      dty